# 3. Data statistics

Tables and figures are generated from reconciled CSV inputs for all five runs. The inputs contain one row per required ID; manifest IDs, per-paper version counts and journal terminal statuses were checked against the filtered snapshot. This is not an independent readback audit of every corpus file. Input hashes and revisions are recorded in `data/m1/final/provenance.json`.

A terminal version can be `ok`, `no_tex` or `unavailable`; terminal coverage is not equivalent to usable TeX or complete reference metadata. `has_source` means at least one version has usable source, not necessarily all versions.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from IPython.display import display, HTML

DATA = Path("../data/m1/final")
FIG = Path("../figures")
FIG.mkdir(exist_ok=True)
df = pd.read_csv(DATA / "papers_stats.csv", dtype={"arxiv_id": str})
assert len(df) == 20_000 and df.arxiv_id.is_unique
ok = (df.n_versions > 0) & (df.n_versions_ok == df.n_versions)

def display_table(frame):
    display(HTML('<div class="table-scroll">' + frame.to_html(index=False, na_rep="N/A") + "</div>"))


## 3.1. Coverage and paper statistics

The counts below use the 20,000 assigned IDs as denominator. A paper with a terminal state can still lack usable TeX. Size and reference averages are per required ID; unavailable reference lists contribute no recorded references, which must not be interpreted as proof of zero references.


In [ ]:
refs_total = df.n_refs_total.sum()
summary = {
    "Required papers": len(df),
    "Papers with all snapshot versions terminal": int(ok.sum()),
    "Terminal paper coverage (%)": 100 * ok.mean(),
    "Papers with at least one version with TeX": int(df.has_source.sum()),
    "Mean recorded size before filtering (bytes / required ID)": df.size_before_bytes.mean(),
    "Mean retained size (bytes / required ID)": df.size_after_bytes.mean(),
    "Reference lists with status ok": int((df.ref_status == "ok").sum()),
    "Reference list status ok (% of required IDs)": 100 * (df.ref_status == "ok").mean(),
    "Retrieved references": int(refs_total),
    "Mean recorded references / required ID": df.n_refs_total.mean(),
    "References with S2 ID (% of retrieved references)": 100 * df.n_refs_meta_ok.sum() / refs_total if refs_total else float("nan"),
    "References with S2 ID, title, authors, date/year (%)": 100 * df.n_refs_meta_complete.sum() / refs_total if refs_total else float("nan"),
}
integer_metrics = {
    "Required papers",
    "Papers with all snapshot versions terminal",
    "Papers with at least one version with TeX",
    "Mean recorded size before filtering (bytes / required ID)",
    "Mean retained size (bytes / required ID)",
    "Reference lists with status ok",
    "Retrieved references",
}
display_values = {
    key: ("N/A" if pd.isna(value) else f"{value:,.0f}" if key in integer_metrics else f"{value:,.2f}")
    for key, value in summary.items()
}
display_table(pd.Series(display_values, name="Value").rename_axis("Metric").reset_index())

display(HTML("<p class='table-caption'>Table 3.1. Coverage, size and reference summary; percentages use the denominators shown.</p>"))


## 3.2. Source size before and after filtering

Sizes are summed from the terminal version records and divided by required IDs in the summary above. They describe the exporter's recorded before-filtering and retained sizes; their meaning can vary by source format and they are not necessarily network transfer bytes. Zero or missing measurements require further audit before treating them as true zero sizes.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(
    [df.size_before_bytes[ok] / 1e6, df.size_after_bytes[ok] / 1e6],
    bins=50,
    log=True,
    label=["Before", "After"],
)
ax.set_xlabel("MB per paper")
ax.set_ylabel("Papers (log scale)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "m1_size_hist.svg")  # vector copy reused on the poster

## 3.3. Versions and references

The first table shows versions per paper. Version terminal states are shown separately below. The reference histogram includes only papers whose reference list status is `ok`; `not_found` is not proof that a paper has no references. Citation-level percentages above use only retrieved references as their denominator.


In [ ]:
version_distribution = df.n_versions.value_counts().sort_index().rename_axis("Versions").reset_index(name="Papers")
display_table(version_distribution)

display(HTML("<p class='table-caption'>Table 3.2. Number of papers by snapshot version count.</p>"))


In [ ]:
refs_ok = df.loc[df.ref_status == "ok", "n_refs_total"]
refs_low = refs_ok.loc[refs_ok <= 150]
refs_high = refs_ok.loc[refs_ok > 150]
assert len(refs_low) + len(refs_high) == len(refs_ok)
fig, (ax_low, ax_high) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
low_bins = list(np.arange(-0.5, 150, 5)) + [150.5]
ax_low.hist(refs_low, bins=low_bins, color="#357a9e", edgecolor="white")
ax_low.set(xlim=(-0.5, 150.5), xlabel="References per paper (0–150)", ylabel="Papers",
           title=f"At most 150 references — {len(refs_low):,} papers")
if len(refs_high):
    upper = np.ceil(refs_high.max() / 20) * 20 + 20.5
    high_bins = np.arange(150.5, upper, 20)
    ax_high.hist(refs_high, bins=high_bins, color="#bd6b35", edgecolor="white")
    ax_high.set_xlim(150.5, high_bins[-1])
else:
    ax_high.text(0.5, 0.5, "No papers above 150", ha="center", transform=ax_high.transAxes)
ax_high.set(xlabel="References per paper (> 150)", ylabel="Papers",
            title=f"More than 150 references — {len(refs_high):,} papers")
fig.savefig(FIG / "m1_refs_hist.svg")


## 3.4. Terminal version and reference-list states

The version counts below come from the last journal record for each expected `(paper, version)`. They describe recorded processing states, not an independent content audit. Reference-list statuses are counted per paper; an `ok` list does not imply every citation has complete metadata.


In [ ]:
version_states = pd.read_csv(DATA / "version_status.csv")
display_table(version_states)
display(HTML("<p class='table-caption'>Table 3.3. Terminal journal states for expected versions across all runs.</p>"))
reference_state_counts = df.ref_status.value_counts(dropna=False).rename_axis("reference_status").reset_index(name="papers")
display_table(reference_state_counts)
display(HTML("<p class='table-caption'>Table 3.4. Reference-list retrieval status, counted by paper across all runs.</p>"))


## 3.5. Version outcomes by run

The stacked bars use versions as the denominator for each run. Absolute counts are displayed in the table: `no_tex` and `unavailable` are terminal journal states, not successfully retrieved TeX. States are reconciled against the last journal record for each expected version.


In [ ]:
version_by_run = pd.read_csv(DATA / "version_status_by_run.csv")
version_counts = version_by_run.pivot(index="run", columns="status", values="versions").fillna(0).astype(int)
version_counts = version_counts.reindex(["only-v1-all", "multi-dang", "multi-hieu", "multi-thien", "multi-tien"])
version_counts = version_counts.reindex(columns=["ok", "no_tex", "unavailable"], fill_value=0)
version_pct = version_counts.div(version_counts.sum(axis=1), axis=0) * 100
ax = version_pct.plot.barh(stacked=True, figsize=(8, 3.5), color=["#337d79", "#e5ad43", "#bd6055"])
ax.set(xlabel="Share of expected versions in run (%)", ylabel="Run", xlim=(0, 100))
ax.invert_yaxis()
ax.legend(title="Journal state", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
display_table(version_counts.assign(total=version_counts.sum(axis=1)).reset_index())

display(HTML("<p class='table-caption'>Table 3.5. Journal version states and total expected versions per run.</p>"))


## 3.6. Reference collection by run

These panels have different denominators: paper reference-list status uses required papers, while citation metadata rates use references actually retrieved. A `not_found` list is not a confirmed list of zero references, and an `ok` list does not imply every reference has complete metadata. The tables retain absolute counts, including small `not_found` counts.


In [ ]:
reference_states = pd.read_csv(DATA / "reference_status_by_run.csv")
reference_meta = pd.read_csv(DATA / "reference_metadata_by_run.csv").set_index("run")
order = ["only-v1-all", "multi-dang", "multi-hieu", "multi-thien", "multi-tien"]
reference_counts = reference_states.pivot(index="run", columns="status", values="papers").fillna(0).astype(int).reindex(order)
reference_pct = reference_counts.div(reference_counts.sum(axis=1), axis=0) * 100
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
reference_pct.plot.barh(stacked=True, ax=ax1, color=["#bf6659", "#337d79"])
ax1.set(xlabel="Share of required papers (%)", ylabel="Run", xlim=(0, 100))
ax1.invert_yaxis()
reference_meta.reindex(order)[["s2_id_pct", "complete_metadata_pct"]].plot.barh(ax=ax2, color=["#537fa5", "#e5ad43"])
ax2.set(xlabel="Share of retrieved references (%)", ylabel="", xlim=(0, 100))
ax2.invert_yaxis()
ax1.legend(title="List status", fontsize=8)
ax2.legend(["Has S2 ID", "S2 ID + title + authors + date/year"], fontsize=8)
fig.tight_layout()
display_table(reference_counts.assign(total=reference_counts.sum(axis=1)).reset_index())
display(HTML("<p class='table-caption'>Table 3.6. Reference-list retrieval status by run; counts are papers.</p>"))
display_table(reference_meta.reindex(order).round(2).reset_index())
display(HTML("<p class='table-caption'>Table 3.7. Metadata availability among retrieved references by run; percentages use retrieved references as the denominator.</p>"))
